# Week 5 Assignment: Personal Knowledge Worker

Build a robust conversational AI that can answer questions about your personal knowledge base.

## Features:
- Ingest documents from your personal folder
- Intelligent LLM-based chunking with headlines and summaries
- Advanced RAG with reranking and query rewriting
- ChromaDB vectorstore for efficient retrieval
- Conversational AI with chat history
- Gradio chat interface

## Step 1: Setup and Configuration

In [4]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
import gradio as gr
import os

load_dotenv(override=True)

# Configuration
MODEL = "gpt-4.1-nano"  # Fast and cost-effective
DB_NAME = "my_knowledge_db"
COLLECTION_NAME = "personal_docs"
EMBEDDING_MODEL = "text-embedding-3-large"
AVERAGE_CHUNK_SIZE = 600  # Slightly larger for comprehensive chunks

# IMPORTANT: Set your personal knowledge base path here!
KNOWLEDGE_BASE_PATH = Path("C:/Users/DELL/KnowledgeBase")  # Change this to your folder!

# Retrieval settings
RETRIEVAL_K = 10  # Number of chunks to retrieve
TOP_K_CONTEXT = 5  # Number of top reranked chunks to use in context

openai = OpenAI()

## Step 2: Define Data Models

Using Pydantic for structured data handling

In [ ]:
class Result(BaseModel):
    """Represents a retrieved chunk with metadata"""
    page_content: str
    metadata: dict

class Chunk(BaseModel):
    """Intelligent chunk with headline, summary, and original text"""
    headline: str = Field(
        description="A brief, searchable heading for this chunk (3-7 words) that captures the main topic"
    )
    summary: str = Field(
        description="A 2-3 sentence summary highlighting key information for Q&A"
    )
    original_text: str = Field(
        description="The exact original text from the document, unchanged"
    )

    def as_result(self, document):
        """Convert chunk to Result format for storage"""
        metadata = {
            "source": document["source"],
            "type": document.get("type", "general")
        }
        # Combine headline, summary, and original text for rich context
        page_content = f"{self.headline}\n\n{self.summary}\n\n{self.original_text}"
        return Result(page_content=page_content, metadata=metadata)

class Chunks(BaseModel):
    """Collection of chunks from a document"""
    chunks: list[Chunk]

class RankOrder(BaseModel):
    """Reranked order of chunk IDs"""
    order: list[int] = Field(
        description="Chunk IDs ordered from most to least relevant"
    )

print("Data models defined")

## Step 3: Document Ingestion

Load documents from your personal knowledge base folder

In [ ]:
def fetch_documents(knowledge_path: Path):
    """
    Load documents from the knowledge base directory.
    Supports: .md, .txt files
    Organizes by subfolder type (if present)
    """
    documents = []
    
    if not knowledge_path.exists():
        print(f" Warning: Path {knowledge_path} does not exist!")
        print(f"Creating example structure at {knowledge_path}")
        knowledge_path.mkdir(parents=True, exist_ok=True)
        
        # Create example file
        example_file = knowledge_path / "example.md"
        example_file.write_text(
            "# Welcome to Your Knowledge Base\n\n"
            "This is an example document. Replace this with your own files!\n\n"
            "## Getting Started\n"
            "1. Add your .md or .txt files to this folder\n"
            "2. Organize them in subfolders if you like\n"
            "3. Run the ingestion process\n"
        )
        print(f"Created example file at {example_file}")
    
    # Check if knowledge_path is a directory or has subdirectories
    if knowledge_path.is_dir():
        # Try to find subdirectories (for organized knowledge bases)
        subdirs = [d for d in knowledge_path.iterdir() if d.is_dir()]
        
        if subdirs:
            # Load from subdirectories with type information
            for folder in subdirs:
                doc_type = folder.name
                for file in folder.rglob("*.md"):
                    try:
                        with open(file, "r", encoding="utf-8") as f:
                            documents.append({
                                "type": doc_type,
                                "source": file.as_posix(),
                                "text": f.read()
                            })
                    except Exception as e:
                        print(f" Error reading {file}: {e}")
                
                for file in folder.rglob("*.txt"):
                    try:
                        with open(file, "r", encoding="utf-8") as f:
                            documents.append({
                                "type": doc_type,
                                "source": file.as_posix(),
                                "text": f.read()
                            })
                    except Exception as e:
                        print(f" Error reading {file}: {e}")
        else:
            # Load from root directory (flat structure)
            for ext in ["*.md", "*.txt"]:
                for file in knowledge_path.glob(ext):
                    try:
                        with open(file, "r", encoding="utf-8") as f:
                            documents.append({
                                "type": "general",
                                "source": file.as_posix(),
                                "text": f.read()
                            })
                    except Exception as e:
                        print(f" Error reading {file}: {e}")
    
    print(f"Loaded {len(documents)} documents from {knowledge_path}")
    return documents

# Load your documents
documents = fetch_documents(KNOWLEDGE_BASE_PATH)

# Show summary
if documents:
    print(f"\nDocument summary:")
    from collections import Counter
    types = Counter([doc["type"] for doc in documents])
    for doc_type, count in types.items():
        print(f"  - {doc_type}: {count} documents")
else:
    print("\n No documents found! Please add files to your knowledge base.")

## Step 4: Intelligent Chunking with LLM

Use an LLM to intelligently split documents and create headlines + summaries

In [ ]:
def make_chunking_prompt(document):
    """Create prompt for LLM to intelligently chunk a document"""
    how_many = max(1, (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1)
    
    return f"""
You are an expert at organizing knowledge bases for optimal retrieval.

Your task is to split this document into overlapping chunks that will be stored in a vector database.
These chunks will be used to answer questions via semantic search.

Document metadata:
- Type: {document["type"]}
- Source: {document["source"]}

Instructions:
1. Split the document into approximately {how_many} chunks (adjust as needed for logical boundaries)
2. Include ~25% overlap between consecutive chunks (about 50-100 words) to ensure context continuity
3. For each chunk, provide:
   - headline: A brief, searchable title (3-7 words) capturing the main topic
   - summary: A 2-3 sentence summary highlighting key information
   - original_text: The exact text from the document (do not modify)
4. Ensure all document content is included across chunks (nothing left out)
5. Chunk at logical boundaries (paragraphs, sections) when possible

Document text:

{document["text"]}

Return the chunks in structured format.
"""

def process_document(document):
    """Process a single document into intelligent chunks"""
    try:
        messages = [{"role": "user", "content": make_chunking_prompt(document)}]
        response = completion(model=MODEL, messages=messages, response_format=Chunks)
        reply = response.choices[0].message.content
        doc_as_chunks = Chunks.model_validate_json(reply).chunks
        return [chunk.as_result(document) for chunk in doc_as_chunks]
    except Exception as e:
        print(f" Error processing document {document['source']}: {e}")
        return []

def create_chunks(documents):
    """Process all documents into chunks with progress bar"""
    chunks = []
    for doc in tqdm(documents, desc="Processing documents"):
        chunks.extend(process_document(doc))
    return chunks

print("Chunking functions ready")

In [ ]:
# Process documents into intelligent chunks
if documents:
    print("Starting intelligent document chunking...")
    chunks = create_chunks(documents)
    print(f"\nCreated {len(chunks)} intelligent chunks from {len(documents)} documents")
else:
    chunks = []
    print(" No documents to process")

## Step 5: Create ChromaDB Vector Store

Store embeddings in ChromaDB for efficient semantic search

In [ ]:
def create_vectorstore(chunks):
    """Create or update ChromaDB vector store with embeddings"""
    if not chunks:
        print(" No chunks to vectorize")
        return None
    
    # Initialize Chroma
    chroma = PersistentClient(path=DB_NAME)
    
    # Delete collection if it exists (fresh start)
    if COLLECTION_NAME in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(COLLECTION_NAME)
        print("Deleted existing collection")
    
    # Extract texts for embedding
    texts = [chunk.page_content for chunk in chunks]
    
    # Create embeddings using OpenAI
    print(f"Creating embeddings for {len(texts)} chunks...")
    emb = openai.embeddings.create(model=EMBEDDING_MODEL, input=texts).data
    vectors = [e.embedding for e in emb]
    
    # Create collection and add vectors
    collection = chroma.get_or_create_collection(COLLECTION_NAME)
    
    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]
    
    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    
    print(f"Vector store created with {collection.count()} documents")
    return collection

# Create the vector store
collection = create_vectorstore(chunks)

## Step 6: Advanced RAG Functions

Implement retrieval, reranking, and query rewriting

In [ ]:
def fetch_context_unranked(question, k=RETRIEVAL_K):
    """Retrieve initial chunks using semantic search"""
    if collection is None:
        return []
    
    # Create query embedding
    query_emb = openai.embeddings.create(model=EMBEDDING_MODEL, input=[question]).data[0].embedding
    
    # Query ChromaDB
    results = collection.query(query_embeddings=[query_emb], n_results=k)
    
    # Convert to Result objects
    chunks = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=doc, metadata=meta))
    
    return chunks

def rerank(question, chunks):
    """Rerank retrieved chunks using LLM for better relevance"""
    if not chunks:
        return chunks
    
    system_prompt = """
You are a document re-ranker for a personal knowledge base.
You receive a question and a list of potentially relevant text chunks.
Your job is to rank these chunks by relevance to the question.
Output only the list of chunk IDs, ordered from most to least relevant.
Include ALL chunk IDs in your ranking.
"""
    
    user_prompt = f"Question: {question}\n\nRank these chunks by relevance:\n\n"
    for i, chunk in enumerate(chunks, 1):
        user_prompt += f"CHUNK {i}:\n{chunk.page_content[:300]}...\n\n"
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    
    try:
        response = completion(model=MODEL, messages=messages, response_format=RankOrder)
        order = RankOrder.model_validate_json(response.choices[0].message.content).order
        return [chunks[i - 1] for i in order if 0 < i <= len(chunks)]
    except Exception as e:
        print(f" Reranking failed: {e}. Using original order.")
        return chunks

def rewrite_query(question, history):
    """Rewrite user question for better retrieval, considering chat history"""
    if not history:
        return question
    
    # Format history for context
    history_text = "\n".join([f"{msg['role']}: {msg['content']}" for msg in history[-4:]])
    
    prompt = f"""
You are helping search a personal knowledge base.

Recent conversation:
{history_text}

Current question: {question}

Rewrite this question as a focused search query that:
1. Incorporates relevant context from the conversation
2. Is specific and keyword-rich
3. Is optimized for semantic search

Output ONLY the rewritten query, nothing else.
"""
    
    try:
        response = completion(model=MODEL, messages=[{"role": "user", "content": prompt}])
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f" Query rewrite failed: {e}. Using original question.")
        return question

def fetch_context(question, history):
    """Retrieve and rerank context for a question"""
    # Rewrite query based on conversation history
    search_query = rewrite_query(question, history)
    if search_query != question:
        print(f"Rewritten query: {search_query}")
    
    # Retrieve initial chunks
    chunks = fetch_context_unranked(search_query)
    
    # Rerank for better relevance
    if len(chunks) > 1:
        chunks = rerank(question, chunks)
    
    # Return top K
    return chunks[:TOP_K_CONTEXT]

print("RAG functions ready")

## Step 7: Question Answering System

Build the conversational AI with memory

In [ ]:
SYSTEM_PROMPT = """
You are a knowledgeable AI assistant helping the user navigate their personal knowledge base.

Your role:
- Answer questions accurately based on the provided context
- Be conversational and helpful
- If information isn't in the context, say so clearly
- Cite sources when relevant (mention the file name)
- Maintain context from the conversation history

Context from knowledge base:
{context}

Answer the user's question based on this context. Be helpful, accurate, and concise.
"""

def make_messages(question, history, chunks):
    """Create message array for LLM with context and history"""
    # Format context from retrieved chunks
    context_parts = []
    for i, chunk in enumerate(chunks, 1):
        source = chunk.metadata.get('source', 'unknown')
        source_name = Path(source).name
        context_parts.append(f"[Source {i}: {source_name}]\n{chunk.page_content}")
    
    context = "\n\n---\n\n".join(context_parts) if context_parts else "No relevant context found."
    
    # Build message list
    messages = [{"role": "system", "content": SYSTEM_PROMPT.format(context=context)}]
    messages.extend(history)
    messages.append({"role": "user", "content": question})
    
    return messages

def answer_question(question, history):
    """Main function to answer a question using RAG"""
    if collection is None:
        return "No knowledge base loaded. Please run the ingestion cells first."
    
    # Fetch relevant context
    chunks = fetch_context(question, history)
    
    # Generate answer
    messages = make_messages(question, history, chunks)
    
    try:
        response = completion(model=MODEL, messages=messages)
        answer = response.choices[0].message.content
        return answer
    except Exception as e:
        return f"Error generating answer: {e}"

print("Q&A system ready")

## Step 8: Test the System

Try asking questions about your knowledge base

In [ ]:
# Test with a sample question
if collection:
    test_question = "What information is in my knowledge base?"
    answer = answer_question(test_question, [])
    print(f"Q: {test_question}")
    print(f"\nA: {answer}")
else:
    print("Please run the ingestion cells first to load your knowledge base")

## Step 9: Gradio Chat Interface

Launch an interactive chat interface

In [ ]:
def chat_interface(message, history):
    """
    Gradio chat function with history support
    
    Args:
        message: Current user message
        history: List of [user_msg, assistant_msg] pairs
    
    Returns:
        Assistant's response
    """
    if collection is None:
        return "No knowledge base loaded. Please run the notebook cells to load your data."
    
    # Convert Gradio history format to OpenAI message format
    chat_history = []
    for user_msg, assistant_msg in history:
        chat_history.append({"role": "user", "content": user_msg})
        chat_history.append({"role": "assistant", "content": assistant_msg})
    
    # Get answer
    response = answer_question(message, chat_history)
    
    return response

print("Chat interface function ready")

In [ ]:
# Create and launch the Gradio interface
if collection:
    demo = gr.ChatInterface(
        fn=chat_interface,
        title="Personal Knowledge Worker",
        description=f"Ask questions about your personal knowledge base ({len(chunks)} chunks from {len(documents)} documents)",
        examples=[
            "What topics are covered in my knowledge base?",
            "Summarize the key information you have access to",
            "What are the main themes in my documents?"
        ],
        theme="soft"
    )
    
    demo.launch()
else:
    print("Cannot launch chat interface - no knowledge base loaded")
    print("Please run the ingestion cells first!")